In [1]:
# ============================================================
# TH Challenge 4 — Books to Scrape
# ============================================================
# Este notebook hace un pipeline completo de datos:
# 1. Scraping de libros y categorías
# 2. Creación de la base de datos
# 3. Enriquecimiento de autores con APIs externas
# ============================================================

import requests
from bs4 import BeautifulSoup
import sqlite3
import time

# ── Configuración general ──────────────────────────────────
BASE_URL = "http://books.toscrape.com/"  # sitio a scrapear
DB_PATH  = "books.db"                    # archivo de base de datos

# Sin User-Agent algunos servidores bloquean la petición
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

# El rating en el HTML viene como texto, lo convertimos a número
RATING_MAP = {
    "One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5
}

print("✅ Librerías importadas y configuración lista")

✅ Librerías importadas y configuración lista


In [ ]:
# ── PARTE 1: WEB SCRAPING ─────────────────────────────────
#
# El scraping funciona en 3 pasos:
# 1. Hacer una petición HTTP GET a la URL
# 2. El servidor responde con HTML
# 3. BeautifulSoup parsea ese HTML para extraer los datos
# ─────────────────────────────────────────────────────────

def obtener_soup(url):
    """
    Hace una petición a la URL y retorna el HTML parseado.
    Si hay cualquier error de red retorna None en vez de detener el programa.
    """
    try:
        respuesta = requests.get(url, headers = HEADERS, timeout=10)
        respuesta.raise_for_status()  # lanza error si status >= 400
        return BeautifulSoup(respuesta.text, "html.parser")
    except Exception as e:
        print(f"Error al acceder a {url}: {e}")
        return None

# Probamos que funciona
soup = obtener_soup(BASE_URL)
print("Título de la página:", soup.title.text.strip())



In [5]:
def obtener_categorias():
    """
    Extrae todas las categorías del índice del sitio.
    Retorna una lista de dicts con nombre, slug y url de cada categoría.
    """
    soup = obtener_soup(BASE_URL)
    
    # Las categorías están en el sidebar izquierdo
    # [1:] descarta el primero que es "Books" (categoría general)
    links = soup.select("ul.nav.nav-list li a")[1:]
    
    categorias = []
    for link in links:
        nombre = link.text.strip()
        href   = link["href"]
        slug   = href.split("/")[-2]        # "mystery_3"
        url    = BASE_URL + href            # URL completa
        
        categorias.append({
            "nombre": nombre,
            "slug":   slug,
            "url":    url
        })
    
    return categorias

# Probamos
categorias = obtener_categorias()
print(f"Total categorías: {len(categorias)}")
print("\nPrimeras 3:")
for cat in categorias[:3]:
    print(f"  - {cat['nombre']} → {cat['url']}")

Total categorías: 50

Primeras 3:
  - Travel → http://books.toscrape.com/catalogue/category/books/travel_2/index.html
  - Mystery → http://books.toscrape.com/catalogue/category/books/mystery_3/index.html
  - Historical Fiction → http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html


In [6]:
def obtener_libros_categoria(url_categoria):
    """
    Extrae todos los libros de una categoría manejando la paginación.
    Sigue el botón "next" hasta que no haya más páginas.
    """
    from urllib.parse import urljoin
    
    libros   = []
    url_actual = url_categoria
    
    while url_actual:
        soup = obtener_soup(url_actual)
        if not soup:
            break
        
        # Cada libro está dentro de <article class="product_pod">
        articulos = soup.select("article.product_pod")
        
        for articulo in articulos:
            # Título — está en el atributo "title" del link
            titulo = articulo.select_one("h3 a")["title"]
            
            # Precio — viene como "£45.17", sacamos el símbolo
            precio_texto = articulo.select_one("p.price_color").text
            precio = float(precio_texto.replace("£", "").replace("Â", "").strip())
            
            # Rating — viene como clase CSS: "star-rating Three"
            # tomamos la segunda clase y la convertimos a número
            rating_texto = articulo.select_one("p.star-rating")["class"][1]
            rating = RATING_MAP.get(rating_texto, 0)
            
            libros.append({
                "titulo": titulo,
                "precio": precio,
                "rating": rating
            })
        
        # Paginación: si hay botón "next" seguimos, sino paramos
        next_btn = soup.select_one("li.next a")
        if next_btn:
            url_actual = urljoin(url_actual, next_btn["href"])
        else:
            url_actual = None
        
        time.sleep(0.5)  # pausa entre páginas (scraping ético)
    
    return libros

# Probamos con Travel
libros_travel = obtener_libros_categoria(categorias[0]["url"])
print(f"Libros en Travel: {len(libros_travel)}")
for libro in libros_travel[:3]:
    print(f"  [{libro['rating']}★] £{libro['precio']} — {libro['titulo']}")

Libros en Travel: 11
  [2★] £45.17 — It's Only the Himalayas
  [4★] £49.43 — Full Moon over Noahâs Ark: An Odyssey to Mount Ararat and Beyond
  [3★] £48.87 — See America: A Celebration of Our National Parks & Treasured Sites


In [7]:
def scrapear_sitio_completo():
    """
    Scrapea todas las categorías y todos los libros del sitio.
    Retorna una lista con todos los libros y su categoría.
    """
    categorias = obtener_categorias()
    todos_los_libros = []
    
    print(f"Scrapeando {len(categorias)} categorías...\n")
    
    for i, cat in enumerate(categorias):
        print(f"[{i+1}/{len(categorias)}] {cat['nombre']}")
        
        libros = obtener_libros_categoria(cat["url"])
        
        # Agregamos el nombre de categoría a cada libro
        for libro in libros:
            libro["categoria"] = cat["nombre"]
            libro["slug"]      = cat["slug"]
        
        todos_los_libros.extend(libros)
        print(f"  → {len(libros)} libros")
        
    print(f"\n✅ Scraping completo: {len(todos_los_libros)} libros")
    return todos_los_libros

# Ejecutar el scraping completo
todos_los_libros = scrapear_sitio_completo()

Scrapeando 50 categorías...

[1/50] Travel
  → 11 libros
[2/50] Mystery
  → 32 libros
[3/50] Historical Fiction
  → 26 libros
[4/50] Sequential Art
  → 75 libros
[5/50] Classics
  → 19 libros
[6/50] Philosophy
  → 11 libros
[7/50] Romance
  → 35 libros
[8/50] Womens Fiction
  → 17 libros
[9/50] Fiction
  → 65 libros
[10/50] Childrens
  → 29 libros
[11/50] Religion
  → 7 libros
[12/50] Nonfiction
  → 110 libros
[13/50] Music
  → 13 libros
[14/50] Default
  → 152 libros
[15/50] Science Fiction
  → 16 libros
[16/50] Sports and Games
  → 5 libros
[17/50] Add a comment
  → 67 libros
[18/50] Fantasy
  → 48 libros
[19/50] New Adult
  → 6 libros
[20/50] Young Adult
  → 54 libros
[21/50] Science
  → 14 libros
[22/50] Poetry
  → 19 libros
[23/50] Paranormal
  → 1 libros
[24/50] Art
  → 8 libros
[25/50] Psychology
  → 7 libros
[26/50] Autobiography
  → 9 libros
[27/50] Parenting
  → 1 libros
[28/50] Adult Fiction
  → 1 libros
[29/50] Humor
  → 10 libros
[30/50] Horror
  → 17 libros
[31/50] Histor

In [8]:
# ── PARTE 2: BASE DE DATOS ────────────────────────────────
#
# Usamos SQLite que viene incluido en Python.
# No necesita instalación ni servidor externo.
# Los datos se guardan en un archivo local: books.db
# ─────────────────────────────────────────────────────────

# Conectamos a la base de datos
# Si el archivo no existe, SQLite lo crea automáticamente
conn   = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# SQLite no activa las foreign keys por defecto
# Esta línea las activa para garantizar integridad referencial
cursor.execute("PRAGMA foreign_keys = ON")

print("✅ Conexión a la base de datos establecida")
print(f"   Archivo: {DB_PATH}")

✅ Conexión a la base de datos establecida
   Archivo: books.db


In [11]:
# Creamos las 4 tablas de la base de datos
# IF NOT EXISTS → si ya existen no falla, las ignora

# Tabla de categorías
# Se crea primero porque books depende de ella
cursor.execute("""
    CREATE TABLE IF NOT EXISTS categories (
        id    INTEGER PRIMARY KEY AUTOINCREMENT,
        name  TEXT NOT NULL UNIQUE,
        slug  TEXT NOT NULL UNIQUE
    )
""")

# Tabla de autores
# Los campos de API pueden ser NULL si no se encontró el autor
cursor.execute("""
    CREATE TABLE IF NOT EXISTS authors (
        id                INTEGER PRIMARY KEY AUTOINCREMENT,
        name              TEXT NOT NULL UNIQUE,
        birth_year        INTEGER,
        country           TEXT,
        external_api_id   TEXT,
        total_known_works INTEGER,
        api_source        TEXT,
        api_status        TEXT DEFAULT 'pending',
        created_at        TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

# Tabla de libros
# CHECK garantiza que el rating siempre esté entre 1 y 5
cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        title       TEXT NOT NULL,
        price       REAL NOT NULL,
        rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
        category_id INTEGER NOT NULL REFERENCES categories(id)
    )
""")

# Tabla intermedia para la relación muchos a muchos entre libros y autores
# PRIMARY KEY compuesta evita duplicados automáticamente
# ON DELETE CASCADE borra las relaciones si se borra un libro o autor
cursor.execute("""
    CREATE TABLE IF NOT EXISTS book_author (
        book_id   INTEGER NOT NULL REFERENCES books(id)   ON DELETE CASCADE,
        author_id INTEGER NOT NULL REFERENCES authors(id) ON DELETE CASCADE,
        PRIMARY KEY (book_id, author_id)
    )
""")

conn.commit()
print("✅ Tablas creadas correctamente")
print("   categories, books, authors, book_author")

✅ Tablas creadas correctamente
   categories, books, authors, book_author


In [12]:
def insertar_categoria(nombre, slug):
    """
    Inserta una categoría y retorna su ID.
    INSERT OR IGNORE → si ya existe no falla, la ignora.
    """
    cursor.execute(
        "INSERT OR IGNORE INTO categories (name, slug) VALUES (?, ?)",
        (nombre, slug)
    )
    conn.commit()
    
    # Recuperamos el ID (sea nuevo o ya existente)
    cursor.execute("SELECT id FROM categories WHERE slug = ?", (slug,))
    return cursor.fetchone()[0]


def insertar_libro(titulo, precio, rating, category_id):
    """
    Inserta un libro y retorna su ID.
    Usamos ? para evitar SQL injection.
    """
    cursor.execute("""
        INSERT INTO books (title, price, rating, category_id)
        VALUES (?, ?, ?, ?)
    """, (titulo, precio, rating, category_id))
    conn.commit()
    return cursor.lastrowid  # ID del registro recién insertado


def cargar_datos_en_db(todos_los_libros):
    """
    Inserta todas las categorías y libros en la base de datos.
    """
    print("Cargando datos en la base de datos...")
    
    for libro in todos_los_libros:
        # Insertar categoría y obtener su ID
        cat_id = insertar_categoria(libro["categoria"], libro["slug"])
        
        # Insertar libro con el ID de su categoría
        insertar_libro(libro["titulo"], libro["precio"], libro["rating"], cat_id)
    
    # Verificar resultados
    cursor.execute("SELECT COUNT(*) FROM books")
    total_libros = cursor.fetchone()[0]
    
    cursor.execute("SELECT COUNT(*) FROM categories")
    total_cats = cursor.fetchone()[0]
    
    print(f"✅ Carga completa")
    print(f"   📚 Libros:     {total_libros}")
    print(f"   📂 Categorías: {total_cats}")

# Ejecutar la carga
cargar_datos_en_db(todos_los_libros)

Cargando datos en la base de datos...
✅ Carga completa
   📚 Libros:     1000
   📂 Categorías: 50


In [13]:
# ── PARTE 3: CONSUMO DE API ───────────────────────────────
#
# Para obtener los autores usamos dos APIs públicas:
# - Open Library → external_api_id y total_known_works
# - Wikipedia    → country y birth_year
#
# Implementamos un cache en memoria para no repetir
# llamadas a la API para el mismo autor.
# ─────────────────────────────────────────────────────────

import re

# Cache en memoria — dura mientras el notebook esté abierto
# Si el autor ya fue consultado, usamos el resultado guardado
cache_autores = {}

print("✅ Cache inicializado")
print(f"   Autores en cache: {len(cache_autores)}")

✅ Cache inicializado
   Autores en cache: 0


In [14]:
def buscar_en_openlibrary(nombre_autor):
    """
    Busca un autor en Open Library.
    Retorna external_api_id y total_known_works.
    """
    try:
        url    = "https://openlibrary.org/search/authors.json"
        params = {"q": nombre_autor, "limit": 1}
        
        respuesta = requests.get(url, params=params, timeout=10)
        data      = respuesta.json()
        
        # Si no encontró ningún autor retornamos None en ambos campos
        if data["numFound"] == 0:
            return {"external_api_id": None, "total_known_works": None}
        
        doc = data["docs"][0]
        return {
            "external_api_id":   doc.get("key"),
            "total_known_works": doc.get("work_count")
        }
    
    except Exception as e:
        print(f"  ⚠️ Error Open Library ({nombre_autor}): {e}")
        return {"external_api_id": None, "total_known_works": None}


# Probamos
resultado = buscar_en_openlibrary("Gillian Flynn")
print(resultado)

{'external_api_id': 'OL1433006A', 'total_known_works': 41}


In [15]:
def buscar_en_wikipedia(nombre_autor):
    """
    Busca la nacionalidad y año de nacimiento de un autor en Wikipedia.
    Retorna country y birth_year extraídos de la descripción.
    """
    try:
        # Wikipedia requiere un User-Agent descriptivo sino devuelve 403
        headers_wiki = {
            "User-Agent": "TH-Challenge4/1.0 (educational project)"
        }
        
        # Wikipedia usa guiones bajos en la URL en vez de espacios
        nombre_url = nombre_autor.replace(" ", "_")
        url        = f"https://en.wikipedia.org/api/rest_v1/page/summary/{nombre_url}"
        
        respuesta  = requests.get(url, headers=headers_wiki, timeout=10)
        
        # Si no encontró la página retornamos None en ambos campos
        if respuesta.status_code != 200:
            return {"country": None, "birth_year": None}
        
        descripcion = respuesta.json().get("description", None)
        
        if not descripcion:
            return {"country": None, "birth_year": None}
        
        # País → primera palabra de la descripción
        # Ejemplo: "American writer (born 1971)" → "American"
        country = descripcion.split()[0]
        
        # Año → primer número de 4 dígitos en la descripción
        # Ejemplo: "American writer (born 1971)" → 1971
        años       = re.findall(r'\d{4}', descripcion)
        birth_year = int(años[0]) if años else None
        
        return {"country": country, "birth_year": birth_year}
    
    except Exception as e:
        print(f"  ⚠️ Error Wikipedia ({nombre_autor}): {e}")
        return {"country": None, "birth_year": None}


# Probamos
resultado = buscar_en_wikipedia("Gillian Flynn")
print(resultado)

{'country': 'American', 'birth_year': 1971}


In [16]:
def obtener_datos_autor(nombre_autor):
    """
    Combina Open Library y Wikipedia para obtener todos los datos del autor.
    Usa cache para no repetir llamadas a la API para el mismo autor.
    """
    # Si ya consultamos este autor, usamos el resultado guardado
    if nombre_autor in cache_autores:
        return cache_autores[nombre_autor]
    
    # Si no está en cache, consultamos las APIs
    datos_ol   = buscar_en_openlibrary(nombre_autor)
    datos_wiki = buscar_en_wikipedia(nombre_autor)
    
    # Determinamos si encontramos datos o no
    if datos_ol["external_api_id"] is None and datos_wiki["country"] is None:
        api_status = "not_found"
    else:
        api_status = "found"
    
    # Combinamos los datos de ambas APIs en un solo dict
    resultado = {
        "external_api_id":   datos_ol["external_api_id"],
        "total_known_works": datos_ol["total_known_works"],
        "country":           datos_wiki["country"],
        "birth_year":        datos_wiki["birth_year"],
        "api_source":        "openlibrary+wikipedia",
        "api_status":        api_status
    }
    
    # Guardamos en cache para la próxima vez
    cache_autores[nombre_autor] = resultado
    
    return resultado


# Probamos
datos = obtener_datos_autor("Gillian Flynn")
print(datos)
print(f"\nAutores en cache: {len(cache_autores)}")

{'external_api_id': 'OL1433006A', 'total_known_works': 41, 'country': 'American', 'birth_year': 1971, 'api_source': 'openlibrary+wikipedia', 'api_status': 'found'}

Autores en cache: 1


In [17]:
def insertar_autor(nombre, datos):
    """
    Inserta un autor en la base de datos y retorna su ID.
    INSERT OR IGNORE → si el autor ya existe no lo duplica.
    """
    cursor.execute("""
        INSERT OR IGNORE INTO authors 
        (name, birth_year, country, external_api_id, total_known_works, api_source, api_status)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        nombre,
        datos["birth_year"],
        datos["country"],
        datos["external_api_id"],
        datos["total_known_works"],
        datos["api_source"],
        datos["api_status"]
    ))
    conn.commit()
    
    # Recuperamos el ID (sea nuevo o ya existente)
    cursor.execute("SELECT id FROM authors WHERE name = ?", (nombre,))
    return cursor.fetchone()[0]


def vincular_libro_autor(book_id, author_id):
    """
    Crea la relación entre un libro y un autor en la tabla book_author.
    INSERT OR IGNORE → evita duplicar la relación si ya existe.
    """
    cursor.execute(
        "INSERT OR IGNORE INTO book_author (book_id, author_id) VALUES (?, ?)",
        (book_id, author_id)
    )
    conn.commit()


# Probamos
datos  = obtener_datos_autor("Gillian Flynn")
aut_id = insertar_autor("Gillian Flynn", datos)
print(f"✅ Autor insertado con ID: {aut_id}")

✅ Autor insertado con ID: 1


In [18]:
def buscar_autor_por_titulo(titulo):
    """
    Busca el autor de un libro por su título en Open Library.
    Limpia el título antes de buscar para mejorar los resultados.
    """
    try:
        # Eliminamos el contenido entre paréntesis del título
        # Ejemplo: "Sharp Objects (Serie #1)" → "Sharp Objects"
        titulo_limpio = re.sub(r'\(.*?\)', '', titulo).strip()
        
        if len(titulo_limpio) < 5:
            return None
        
        url    = "https://openlibrary.org/search.json"
        params = {"title": titulo_limpio, "limit": 1}
        
        respuesta = requests.get(url, params=params, timeout=10)
        data      = respuesta.json()
        
        if data["numFound"] == 0:
            return None
        
        autores = data["docs"][0].get("author_name")
        return autores[0] if autores else None
    
    except Exception as e:
        print(f"  ⚠️ Error buscando título ({titulo[:30]}): {e}")
        return None


# Pipeline completo de enriquecimiento
def enriquecer_autores():
    """
    Para cada libro en la DB:
    1. Busca el autor por título en Open Library
    2. Obtiene los datos del autor con las APIs
    3. Inserta el autor y lo vincula al libro
    """
    cursor.execute("SELECT id, title FROM books")
    libros = cursor.fetchall()
    
    encontrados    = 0
    no_encontrados = 0
    
    print(f"Enriqueciendo {len(libros)} libros...\n")
    
    for i, (book_id, titulo) in enumerate(libros):
        if i % 100 == 0:
            print(f"  [{i}/{len(libros)}] procesando...")
        
        # Paso 1: buscar autor por título
        nombre_autor = buscar_autor_por_titulo(titulo)
        
        if not nombre_autor:
            # No encontró autor → guardamos como Unknown con NULLs
            datos = {
                "birth_year": None, "country": None,
                "external_api_id": None, "total_known_works": None,
                "api_source": "openlibrary+wikipedia", "api_status": "not_found"
            }
            nombre_autor = "Unknown"
            no_encontrados += 1
        else:
            # Paso 2: obtener datos completos del autor (con cache)
            datos = obtener_datos_autor(nombre_autor)
            encontrados += 1
        
        # Paso 3: insertar autor y vincular al libro
        author_id = insertar_autor(nombre_autor, datos)
        vincular_libro_autor(book_id, author_id)
        
        time.sleep(0.3)
    
    total = encontrados + no_encontrados
    print(f"\n✅ Enriquecimiento completo")
    print(f"   Encontrados:    {encontrados} ({encontrados/total*100:.1f}%)")
    print(f"   No encontrados: {no_encontrados} ({no_encontrados/total*100:.1f}%)")

# Ejecutar
enriquecer_autores()

Enriqueciendo 1000 libros...

  [0/1000] procesando...
  [100/1000] procesando...
  ⚠️ Error buscando título (Giant Days, Vol. 1 (Giant Days): HTTPSConnectionPool(host='openlibrary.org', port=443): Max retries exceeded with url: /search.json?title=Giant+Days%2C+Vol.+1&limit=1 (Caused by ConnectTimeoutError(<HTTPSConnection(host='openlibrary.org', port=443) at 0x285c1194dd0>, 'Connection to openlibrary.org timed out. (connect timeout=10)'))
  ⚠️ Error Open Library (Louisa May Alcott): HTTPSConnectionPool(host='openlibrary.org', port=443): Max retries exceeded with url: /search/authors.json?q=Louisa+May+Alcott&limit=1 (Caused by ConnectTimeoutError(<HTTPSConnection(host='openlibrary.org', port=443) at 0x285c0bf3e30>, 'Connection to openlibrary.org timed out. (connect timeout=10)'))
  ⚠️ Error buscando título (Dirty (Dive Bar #1)): HTTPSConnectionPool(host='openlibrary.org', port=443): Max retries exceeded with url: /search.json?title=Dirty&limit=1 (Caused by ConnectTimeoutError(<HTTPSCon

In [19]:
cursor.execute("SELECT COUNT(*) FROM authors")
print(f"Autores:    {cursor.fetchone()[0]}")

cursor.execute("SELECT COUNT(*) FROM book_author")
print(f"Relaciones: {cursor.fetchone()[0]}")

cursor.execute("SELECT COUNT(*) FROM authors WHERE api_status = 'found'")
print(f"Encontrados: {cursor.fetchone()[0]}")

cursor.execute("SELECT COUNT(*) FROM authors WHERE api_status = 'not_found'")
print(f"No encontrados: {cursor.fetchone()[0]}")

Autores:    572
Relaciones: 1000
Encontrados: 571
No encontrados: 1


In [20]:
# ── PARTE 4: CONSULTAS SQL ────────────────────────────────
#
# Ahora que tenemos los datos cargados hacemos consultas
# para extraer información útil de la base de datos.
# ─────────────────────────────────────────────────────────

print("Base de datos lista para consultas")
cursor.execute("SELECT COUNT(*) FROM books")
print(f"  📚 Libros:     {cursor.fetchone()[0]}")
cursor.execute("SELECT COUNT(*) FROM authors")
print(f"  ✍️  Autores:    {cursor.fetchone()[0]}")
cursor.execute("SELECT COUNT(*) FROM categories")
print(f"  📂 Categorías: {cursor.fetchone()[0]}")

Base de datos lista para consultas
  📚 Libros:     1000
  ✍️  Autores:    572
  📂 Categorías: 50


In [21]:
# CONSULTA 1: Libros con buen rating y precio accesible
# Usamos JOIN para obtener el nombre de la categoría
# WHERE filtra por dos condiciones simultáneamente

cursor.execute("""
    SELECT 
        b.title,
        b.price,
        b.rating,
        c.name AS categoria
    FROM books b
    JOIN categories c ON b.category_id = c.id
    WHERE b.rating > 3
      AND b.price < 20.0
    ORDER BY b.rating DESC, b.price ASC
    LIMIT 10
""")

resultados = cursor.fetchall()

print(f"Libros con rating > 3 y precio < £20: {len(resultados)} resultados")
print()
for row in resultados:
    print(f"  [{row[2]}★] £{row[1]:.2f} — {row[0][:45]} ({row[3]})")

Libros con rating > 3 y precio < £20: 10 resultados

  [5★] £10.00 — An Abundance of Katherines (Young Adult)
  [5★] £10.23 — Greek Mythic History (Default)
  [5★] £11.05 — The Power Greens Cookbook: 140 Delicious Supe (Food and Drink)
  [5★] £11.21 — Dear Mr. Knightley (Fiction)
  [5★] £11.33 — The Darkest Corners (Young Adult)
  [5★] £11.38 — Naturally Lean: 125 Nourishing Gluten-Free, P (Food and Drink)
  [5★] £11.64 — Fruits Basket, Vol. 2 (Fruits Basket #2) (Sequential Art)
  [5★] £11.83 — Old School (Diary of a Wimpy Kid #10) (Humor)
  [5★] £11.89 — Superman Vol. 1: Before Truth (Superman by Ge (Sequential Art)
  [5★] £12.16 — Every Heart a Doorway (Every Heart A Doorway  (Fantasy)


In [23]:
# CONSULTA 2: Categoría con mayor precio promedio
# GROUP BY agrupa todos los libros de cada categoría
# AVG calcula el promedio del grupo

cursor.execute("""
    SELECT 
        c.name AS categoria,
        COUNT(b.id) AS total_libros,
        ROUND(AVG(b.price), 2) AS precio_promedio
    FROM categories c
    JOIN books b ON c.id = b.category_id
    GROUP BY c.id, c.name
    ORDER BY precio_promedio DESC
    LIMIT 10
""")

resultados = cursor.fetchall()

print("Top 10 categorías por precio promedio:")
print()
for row in resultados:
    print(f"  £{row[2]:.2f} — {row[0]} ({row[1]} libros)")

Top 10 categorías por precio promedio:

  £58.33 — Suspense (1 libros)
  £54.81 — Novels (1 libros)
  £53.61 — Politics (3 libros)
  £51.45 — Health (4 libros)
  £46.38 — New Adult (6 libros)
  £42.50 — Christian (3 libros)
  £41.17 — Sports and Games (5 libros)
  £40.62 — Self Help (5 libros)
  £39.79 — Travel (11 libros)
  £39.59 — Fantasy (48 libros)


In [24]:
# CONSULTA 3: Autor con peor promedio de rating
# HAVING filtra grupos — no podemos usar WHERE porque
# COUNT se calcula después del GROUP BY
# Atravesamos la relación M:N: authors → book_author → books

cursor.execute("""
    SELECT 
        a.name AS autor,
        COUNT(b.id) AS total_libros,
        ROUND(AVG(b.rating), 2) AS promedio_rating
    FROM authors a
    JOIN book_author ba ON a.id = ba.author_id
    JOIN books b ON ba.book_id = b.id
    WHERE a.name != 'Unknown'
    GROUP BY a.id, a.name
    HAVING COUNT(b.id) >= 5
    ORDER BY promedio_rating ASC
    LIMIT 5
""")

resultados = cursor.fetchall()

print("Autores con peor promedio de rating (mínimo 5 libros):")
print()
for row in resultados:
    print(f"  {row[2]:.2f}★ — {row[0]} ({row[1]} libros)")

Autores con peor promedio de rating (mínimo 5 libros):

  2.25★ — Sophie Kinsella (8 libros)
  2.30★ — Worth Books (10 libros)
  2.57★ — J. K. Rowling (7 libros)
  2.77★ — Stephen King (13 libros)
  2.86★ — Cassandra Clare (7 libros)


In [25]:
# CONSULTA 4: Top 5 autores con más libros
# Atravesamos la relación M:N para contar libros por autor
# Excluimos "Unknown" para mostrar solo autores reales

cursor.execute("""
    SELECT 
        a.name AS autor,
        a.country AS pais,
        COUNT(b.id) AS total_libros,
        ROUND(AVG(b.rating), 2) AS rating_promedio
    FROM authors a
    JOIN book_author ba ON a.id = ba.author_id
    JOIN books b ON ba.book_id = b.id
    WHERE a.name != 'Unknown'
    GROUP BY a.id, a.name
    ORDER BY total_libros DESC
    LIMIT 5
""")

resultados = cursor.fetchall()

print("Top 5 autores con más libros:")
print()
for row in resultados:
    pais = row[1] if row[1] else "N/D"
    print(f"  {row[2]} libros — {row[0]} ({pais}) — {row[3]:.2f}★")

Top 5 autores con más libros:

  13 libros — Stephen King (American) — 2.77★
  10 libros — Worth Books (N/D) — 2.30★
  8 libros — Sophie Kinsella (English) — 2.25★
  7 libros — J. K. Rowling (British) — 2.57★
  7 libros — Cassandra Clare (American) — 2.86★


In [26]:
# CONSULTA 5 (OBLIGATORIA): País con más libros de rating > 3
# Esta consulta requiere la API — sin country no existe
# Traversamos 3 tablas: books → book_author → authors

cursor.execute("""
    SELECT 
        a.country AS pais,
        COUNT(b.id) AS total_libros,
        ROUND(AVG(b.rating), 2) AS rating_promedio
    FROM books b
    JOIN book_author ba ON b.id = ba.book_id
    JOIN authors a ON ba.author_id = a.id
    WHERE b.rating > 3
      AND a.country IS NOT NULL
      AND a.name != 'Unknown'
    GROUP BY a.country
    ORDER BY total_libros DESC
    LIMIT 10
""")

resultados = cursor.fetchall()

print("País con más libros de rating > 3:")
print()
for row in resultados:
    print(f"  {row[1]} libros — {row[0]} — {row[2]:.2f}★ promedio")

País con más libros de rating > 3:

  114 libros — American — 4.54★ promedio
  18 libros — English — 4.44★ promedio
  17 libros — British — 4.41★ promedio
  8 libros — Topics — 4.75★ promedio
  8 libros — Canadian — 4.38★ promedio
  3 libros — Italian — 4.33★ promedio
  3 libros — Australian — 4.67★ promedio
  2 libros — Irish — 4.50★ promedio
  2 libros — French — 5.00★ promedio
  1 libros — Spanish — 5.00★ promedio


In [27]:
# SUBCONSULTA: Libros con precio mayor al promedio de su categoría
# Una subconsulta es un SELECT dentro de otro SELECT
# La subconsulta interna calcula el promedio de cada categoría
# La subconsulta externa filtra los libros que superan ese promedio

cursor.execute("""
    SELECT 
        b.title,
        b.price,
        c.name AS categoria,
        (SELECT ROUND(AVG(b2.price), 2) 
         FROM books b2 
         WHERE b2.category_id = b.category_id) AS promedio_categoria
    FROM books b
    JOIN categories c ON b.category_id = c.id
    WHERE b.price > (
        SELECT AVG(b3.price)
        FROM books b3
        WHERE b3.category_id = b.category_id
    )
    ORDER BY b.price DESC
    LIMIT 10
""")

resultados = cursor.fetchall()

print("Libros más caros que el promedio de su categoría:")
print()
for row in resultados:
    print(f"  £{row[1]:.2f} (promedio: £{row[3]}) — {row[0][:40]} ({row[2]})")

Libros más caros que el promedio de su categoría:

  £59.99 (promedio: £33.93) — The Perfect Play (Play by Play #1) (Romance)
  £59.98 (promedio: £36.07) — Last One Home (New Beginnings #1) (Fiction)
  £59.95 (promedio: £34.22) — Civilization and Its Discontents (Psychology)
  £59.92 (promedio: £31.41) — The Barefoot Contessa Cookbook (Food and Drink)
  £59.90 (promedio: £34.26) — The Diary of a Young Girl (Nonfiction)
  £59.71 (promedio: £31.43) — The Bone Hunters (Lexy Vaughan & Steven  (Thriller)
  £59.64 (promedio: £37.29) — Thomas Jefferson and the Tripoli Pirates (History)
  £59.48 (promedio: £31.72) — Boar Island (Anna Pigeon #19) (Mystery)
  £59.45 (promedio: £36.07) — The Improbability of Love (Fiction)
  £59.45 (promedio: £34.26) — The Man Who Mistook His Wife for a Hat a (Nonfiction)


In [28]:
# FUNCIÓN DE VENTANA: Ranking de libros por precio dentro de cada categoría
# RANK() OVER (PARTITION BY) numera los libros dentro de cada categoría
# A diferencia de GROUP BY, mantiene todas las filas individuales
# Usamos subconsulta porque SQLite no permite filtrar por alias de ventana

cursor.execute("""
    SELECT title, price, categoria, ranking
    FROM (
        SELECT 
            b.title,
            b.price,
            c.name AS categoria,
            RANK() OVER (PARTITION BY b.category_id ORDER BY b.price DESC) AS ranking
        FROM books b
        JOIN categories c ON b.category_id = c.id
    )
    WHERE ranking = 1
    ORDER BY price DESC
    LIMIT 10
""")

resultados = cursor.fetchall()

print("Libro más caro por categoría:")
print()
for row in resultados:
    print(f"  #{row[3]} £{row[1]:.2f} — {row[0][:40]} ({row[2]})")

Libro más caro por categoría:

  #1 £59.99 — The Perfect Play (Play by Play #1) (Romance)
  #1 £59.98 — Last One Home (New Beginnings #1) (Fiction)
  #1 £59.95 — Civilization and Its Discontents (Psychology)
  #1 £59.92 — The Barefoot Contessa Cookbook (Food and Drink)
  #1 £59.90 — The Diary of a Young Girl (Nonfiction)
  #1 £59.71 — The Bone Hunters (Lexy Vaughan & Steven  (Thriller)
  #1 £59.64 — Thomas Jefferson and the Tripoli Pirates (History)
  #1 £59.48 — Boar Island (Anna Pigeon #19) (Mystery)
  #1 £59.15 — The Gray Rhino: How to Recognize and Act (Add a comment)
  #1 £59.04 — Life Without a Recipe (Autobiography)


In [ ]:
# ── PARTE 5: INDEXACIÓN Y PERFORMANCE ────────────────────
#
# Un índice es una estructura que acelera las búsquedas.
# Sin índice → Full Table Scan (lee todos los registros)
# Con índice → va directo a los registros relevantes
# ─────────────────────────────────────────────────────────

import time

# Consulta que vamos a medir
consulta = """
    SELECT b.title, b.price, b.rating, c.name
    FROM books b
    JOIN categories c ON b.category_id = c.id
    WHERE b.rating > 3
      AND b.price < 40.0
    ORDER BY b.price ASC
"""

# Medimos SIN índice
tiempos = []
for i in range(3):
    inicio = time.perf_counter()
    cursor.execute(consulta)
    cursor.fetchall()
    fin = time.perf_counter()
    tiempos.append((fin - inicio) * 1000)

tiempo_sin = sum(tiempos) / len(tiempos)
print(f"Sin índice:  {tiempo_sin:.3f} ms")

# Creamos el índice
cursor.execute("CREATE INDEX IF NOT EXISTS idx_books_rating ON books(rating)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_books_price  ON books(price)")
conn.commit()
print("\n✅ Índices creados")

# Medimos CON índice
tiempos = []
for i in range(3):
    inicio = time.perf_counter()
    cursor.execute(consulta)
    cursor.fetchall()
    fin = time.perf_counter()
    tiempos.append((fin - inicio) * 1000)

tiempo_con = sum(tiempos) / len(tiempos)
print(f"Con índice:  {tiempo_con:.3f} ms")

mejora = ((tiempo_sin - tiempo_con) / tiempo_sin) * 100
print(f"\n📊 Mejora: {mejora:.1f}%")